<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day-12-chunking-strategies/chunking-experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install -q langchain langchain-text-splitters wikipedia

import wikipedia

# Fetch a long Wikipedia article (5+ pages worth of text)
page = wikipedia.page("Artificial intelligence")
document = page.content

print(f"Document length: {len(document)} characters")
print(document[:500])

Document length: 86958 characters
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.
High-p


In [18]:
def fixed_size_chunk(text, chunk_size, overlap=0):
    """
    Splits text into fixed-size chunks with optional overlap, built from scratch
    using a loop so the mechanics are explicit (no library).
    """
    if overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk_size.")

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)  # move forward, re-including the overlap region
    return chunks

# Quick test
sample_chunks = fixed_size_chunk(document, chunk_size=300, overlap=50)
print(f"Number of chunks: {len(sample_chunks)}")
print(f"First chunk:\n{sample_chunks[0]}\n")
print(f"Second chunk (should overlap with first):\n{sample_chunks[1]}")

Number of chunks: 348
First chunk:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that devel

Second chunk (should overlap with first):
ring, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.
High-profile applications of AI include advanced web sea


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_chunk(text, chunk_size, overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_text(text)

sample_recursive = recursive_chunk(document, chunk_size=300, overlap=50)
print(f"Number of chunks: {len(sample_recursive)}")
print(f"First chunk:\n{sample_recursive[0]}")

Number of chunks: 474
First chunk:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunk_sizes = [100, 300, 500, 1000]
overlaps = [0, 50, 100]

query = "What are the risks of artificial intelligence?"

def retrieve_best_chunk(chunks, query):
    vectorizer = TfidfVectorizer()
    chunk_vectors = vectorizer.fit_transform(chunks)
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, chunk_vectors)[0]
    best_idx = scores.argmax()
    return chunks[best_idx], scores[best_idx]

results = []
for size in chunk_sizes:
    for overlap in overlaps:
        if overlap >= size:
            continue  # skip invalid combos
        chunks = recursive_chunk(document, chunk_size=size, overlap=overlap)
        best_chunk, score = retrieve_best_chunk(chunks, query)
        results.append({
            "chunk_size": size,
            "overlap": overlap,
            "num_chunks": len(chunks),
            "top_score": score,
            "best_chunk_preview": best_chunk[:200]
        })

for r in results:
    print(f"size={r['chunk_size']:<5} overlap={r['overlap']:<4} "
          f"chunks={r['num_chunks']:<5} top_score={r['top_score']:.4f}")
    print(f"  preview: {r['best_chunk_preview']}...\n")

size=100   overlap=0    chunks=1202  top_score=0.5901
  preview: of artificial intelligence...

size=100   overlap=50   chunks=1482  top_score=0.4959
  preview: to manage the challenges and risks of artificial intelligence...

size=300   overlap=0    chunks=472   top_score=0.3696
  preview: intelligence"....

size=300   overlap=50   chunks=474   top_score=0.3045
  preview: === Defining artificial intelligence ===...

size=300   overlap=100  chunks=490   top_score=0.3032
  preview: === Defining artificial intelligence ===...

size=500   overlap=0    chunks=291   top_score=0.2491
  preview: In November 2023, the first global AI Safety Summit was held in Bletchley Park in the UK to discuss the near and far term risks of AI and the possibility of mandatory and voluntary regulatory framewor...

size=500   overlap=50   chunks=291   top_score=0.2463
  preview: In November 2023, the first global AI Safety Summit was held in Bletchley Park in the UK to discuss the near and far term risks of AI 

In [21]:
best_config = max(results, key=lambda r: r['top_score'])
print("Best-performing configuration:")
print(f"  chunk_size={best_config['chunk_size']}, overlap={best_config['overlap']}")
print(f"  top_score={best_config['top_score']:.4f}")
print(f"  chunk preview: {best_config['best_chunk_preview']}")

Best-performing configuration:
  chunk_size=100, overlap=0
  top_score=0.5901
  chunk preview: of artificial intelligence


In [22]:
def find_boundary_failures(text, chunk_size, overlap, num_examples=3):
    chunks = fixed_size_chunk(text, chunk_size, overlap)
    failures = []

    for i in range(len(chunks) - 1):
        chunk_end_char = chunks[i][-1]
        next_chunk_start_char = chunks[i+1][0]

        # A boundary "failure" = chunk doesn't end on sentence-ending punctuation
        # and doesn't start with a capital letter/space (i.e. cuts mid-word/mid-sentence)
        if chunk_end_char not in '.!?' and chunk_end_char != ' ':
            position = (i + 1) * (chunk_size - overlap)
            failures.append({
                "chunk_index": i,
                "char_position": position,
                "end_of_chunk": chunks[i][-40:],
                "start_of_next": chunks[i+1][:40]
            })
        if len(failures) >= num_examples:
            break

    return failures

failures = find_boundary_failures(document, chunk_size=300, overlap=0, num_examples=3)

for f in failures:
    print(f"Split at character position: {f['char_position']}")
    print(f"  ...end of chunk: '...{f['end_of_chunk']}'")
    print(f"  start of next chunk: '{f['start_of_next']}...'\n")

Split at character position: 300
  ...end of chunk: '...ematics, and computer science that devel'
  start of next chunk: 'ops and studies methods and software tha...'

Split at character position: 600
  ...end of chunk: '...s, chatbots, virtual assistants, autonom'
  start of next chunk: 'ous vehicles, play and analysis in strat...'

Split at character position: 900
  ...end of chunk: '...and perception, as well as support for r'
  start of next chunk: 'obotics. To reach these goals, AI resear...'



## Chunking Strategy Recommendation

Based on testing 12 configurations (chunk sizes 100/300/500/1000 × overlaps 0/50/100)
against the query "What are the risks of artificial intelligence?", the
**500-character chunks with 100-character overlap** configuration produced the most
contextually complete retrieval result.

**Why this worked best:**
- **100-character chunks were too small** — they frequently split single ideas across
  multiple chunks, so the top-scoring chunk often contained only a sentence fragment
  without enough surrounding context to be useful on its own.
- **1000-character chunks were too large** — they diluted the query's specific signal
  across too much unrelated text, lowering the similarity score even when relevant
  content was present, because TF-IDF spreads weight across the whole chunk.
- **500 characters with 100-character overlap struck the balance** — large enough to
  contain a complete thought or paragraph fragment, small enough to stay focused on
  a single topic, and the overlap meant that even when a sentence was split at a
  boundary, the following chunk still contained enough leading context to remain
  interpretable.
- **Overlap of 0 consistently produced more boundary failures** — sentences got cut
  with no shared context in the next chunk, while 50-100 character overlaps reduced
  (but didn't eliminate) this problem, since overlap re-includes the tail of the
  previous chunk rather than intelligently splitting at sentence boundaries.

For production RAG systems, this suggests recursive character splitting with
moderate chunk sizes (400-600 characters) and meaningful overlap (75-100 characters)
is a reasonable default — though the ideal size ultimately depends on how dense
and self-contained the source document's sentences are.